# Fourier-Morse Fit with Configurable Weighting, Interpolation & Interaction

Consolidates the earlier fit notebooks (08-11) into a single parameterised
notebook. The interaction (`interaction`), the alpha-weight scheme, and the
energy-minimum interpolation can be changed in the configuration cell. The
defaults are **Poisson weighting with $\lambda = 0.8$** and **harmonic
interpolation enabled** (`interpolate = True`, off-grid $r_e$); the interaction
defaults to OA.

Harmonic ceilings are taken from `harmonic_ceils` (OA/OP $(20,1)$, EP/EA $(8,1)$)
to match the baseline used by ``plot_oa_error.py``. The fitted energy surface is
exported to a CSV that can be fed to ``plot_oa_error.py --fit-input``.

In [ ]:
from pathlib import Path

from chimorse.config import load_molecule_info
from chimorse.datasets import ensure_reference_data
from chimorse.dataio import load_data
from chimorse.fitting import generate_fourier_morse_data, make_weight_func

In [ ]:
# ---- molecule / interaction ----
molecule_name = 'PA'
interaction   = 'OA'
zero_zeta     = True
alpha_fit     = True
harmonic_ceils = {'EP': (8, 1), 'EA': (8, 1), 'OP': (20, 1), 'OA': (20, 1)}

# ---- fit configuration (change these) ----
weight_scheme = 'poisson'      # 'equal' | 'gaussian' | 'poisson' | 'energy'
weight_lam    = 0.8            # poisson lambda (gaussian sigma / energy exponent)
energy_eps    = 1e-4           # floor for the energy weight
interpolate   = True           # off-grid harmonic interpolation of r_e (True/False)

# ---- build the alpha-fit weight function ----
weight_func = make_weight_func(weight_scheme, lam=weight_lam, eps=energy_eps)
tag = f"w_{weight_scheme}_lam{weight_lam}"
print('interaction   :', interaction)
print('weight_scheme :', weight_scheme)
print('interpolate   :', interpolate)
print('tag           :', tag)

In [ ]:
data_root = Path('../data')
data_dir = ensure_reference_data(molecule_name, data_root=data_root)
molecule = load_molecule_info(molecule_name, metadata_path=data_dir / 'metadata.json')
df = load_data(molecule, interaction, zero_zeta=zero_zeta)
print(f'{molecule.name} {interaction}: {len(df)} rows')

In [ ]:
df_model = generate_fourier_morse_data(
    df, molecule, interaction, harmonic_ceils,
    alpha_fit=alpha_fit, interpolate=interpolate, weight_func=weight_func,
    print_errors=True, near_eq_delta_r=.5,
)
print('model rows:', len(df_model))

In [ ]:
out_csv = data_dir / f'df_model_{interaction}_{tag}.csv'
df_model.to_csv(out_csv, index=False)
print('saved fit:', out_csv)

### Next step

Run ``plot_oa_error.py --fit-input <out_csv> --tag <tag>``
to generate the corresponding error plots.